In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
REUSE_VESSELNESS = True
REBUILD_REPORT = True


# OpenPlaque — Global Left-Coronary Graph Reconstruction

The validated ~25 mm LAD is the only positive coronary anchor. The validated RCA is a hard 3 mm exclusion corridor and is never a target. Aorta geometry is used only for broad ROI construction and post-hoc ranking, never for node discovery, edge cost, or path tracing.


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --depth 1 --branch lad-global-left-coronary-graph-from-main https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install numpy pandas scipy matplotlib SimpleITK scikit-image
import sys
sys.path.insert(0,'/content/OpenPlaque/src')
!git -C /content/OpenPlaque rev-parse HEAD


In [ ]:
from IPython.display import display, Image
from openplaque.lad_global_left_coronary_graph import GlobalLeftCoronaryGraphWorkflow, synthetic_global_graph_self_test
test=synthetic_global_graph_self_test(); display(test); assert test['passed'], test
wf=GlobalLeftCoronaryGraphWorkflow(root='/content/drive/MyDrive/OpenPlaque', reuse=REUSE_VESSELNESS)


In [ ]:
prov=wf.load_inputs(); display(prov)
print('Validated LAD length:', prov['lad_length_mm'])
print('CT cache:', prov['ct_cache'])


In [ ]:
wf.build_vesselness()
print('ROI source bounds:', wf.roi_lo, wf.roi_hi)
print('Isotropic vesselness shape:', wf.vessel.shape)


In [ ]:
nodes=wf.discover_nodes(); print('Graph nodes:', len(nodes)); display(nodes.sort_values('plane_score',ascending=False).head(30))
edges=wf.build_graph(); print('Graph edges:', len(edges)); display(edges.head(30))


In [ ]:
paths=wf.enumerate_paths(); print('Candidate graph paths:', len(paths)); display(paths)
summary=wf.validate_paths(); display(summary)
display(wf.validation)


In [ ]:
names=wf.make_figures()
for name in names:
    print(name)
    display(Image(filename=str(wf.out/name)))


In [ ]:
report,zip_path=wf.package()
print('STATUS:', wf.summary['status'])
print('HTML:', report)
print('Final ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LAD_GLOBAL_LEFT_CORONARY_GRAPH_REPORT_BACK.zip')
